# ENSO index data preparation

This notebook prepares ONI, RONI and TNI records for the ENSO analysis. It loads manually downloaded source files from the University of Miami and NOAA PSL, validates their monthly records, and combines them by date.

Validation checks required columns, numeric values, first-of-month dates and duplicate records. Identical rows are removed and reported; conflicting monthly records stop processing. Full source history and missing values are retained.

RONI is the project's primary index, ONI is secondary, and TNI is retained as provided. ONI and RONI represent centered three-month values; their monthly timestamps are alignment keys.

Inputs are stored under ../indicies_data/first_pass:
- ONI_NINO34_v6.txt
- RONI_NINO34_v6.txt
- tni.long.csv

Outputs:
- ../indicies_data/first_pass/enso_historical_indicies.csv
- ../outputs/enso_index_source_inventory.csv

The source inventory records file versions, URLs, checksums, local timestamps and usable coverage.

This notebook runs independently of station downloading and splicing. ENSO classification, seasonal analysis and the 1950-onward results filter are handled downstream in the table builder.

In [ ]:
#dependencies
from pathlib import Path
import pandas as pd
from hashlib import sha256
from urllib.request import Request, urlopen

In [2]:
#makes data directory structure
indicies_dir=Path("../indicies_data/first_pass")
indicies_dir.mkdir(parents=True, exist_ok=True)

# save name of final product
output_roni_oni_tni_indicies= indicies_dir/'enso_historical_indicies.csv'

In [ ]:
# cache the official CPC tables used for 1950-onward values
refresh_cpc = False

cpc_urls = {
    "ONI": (
        "https://www.cpc.ncep.noaa.gov/"
        "products/analysis_monitoring/enso/oni/v6/"
    ),
    "RONI": (
        "https://www.cpc.ncep.noaa.gov/"
        "products/analysis_monitoring/enso/roni/"
    ),
}

cpc_paths = {
    name: indicies_dir / f"{name}_CPC_v6.html"
    for name in cpc_urls
}

for index_name, url in cpc_urls.items():
    path = cpc_paths[index_name]

    if refresh_cpc or not path.exists():
        request = Request(
            url,
            headers={"User-Agent": "ENSO-research/1.1"},
        )
        with urlopen(request, timeout=60) as response:
            content = response.read()

        html = content.decode("utf-8")
        if "ERSSTv6" not in html or "<table" not in html.lower():
            raise ValueError(
                f"{index_name}: response is not the expected CPC table."
            )

        path.write_bytes(content)
        print(f"{index_name}: downloaded {path.name}")
    else:
        print(f"{index_name}: using cached {path.name}")

In [ ]:
#get the TNI from NOAA and historic (pre 19050) from University of Miami into dataframes 

'''
direct web->notebook access encountered certification errors I could not resolve 
this data was manually placed into ../indicies_data/first_pass

oni_historic        ->     https://bmcnoldy.earth.miami.edu/tropics/oni/ONI_NINO34_v6.txt
tni_historic_csv    ->     https://psl.noaa.gov/data/timeseries/month/data/tni.long.csv
tni_historic_nc     ->     https://psl.noaa.gov/data/timeseries/month/data/tni.long.nc (this set may not be used in the workflow)
roni_historic       ->     https://bmcnoldy.earth.miami.edu/tropics/roni/RONI_NINO34_v6.txt
'''



#assign file paths
oni_historic=Path(indicies_dir/'ONI_NINO34_v6.txt')
roni_historic=Path(indicies_dir/'RONI_NINO34_v6.txt')
tni_historic=Path(indicies_dir/'tni.long.csv')

#read txt/csv files and init their variables

#oni
oni_historic_df=pd.read_csv(
    oni_historic,
    skiprows=10,
    sep=r"\s+",
    na_values=[-99.99],
)

#roni
roni_historic_df=pd.read_csv(
    roni_historic,
    skiprows=10,
    sep=r"\s+",
    na_values=[-99.99],
    )

#tni
tni_historic_df = pd.read_csv(
    tni_historic,
    header=0,
    names=["Date", "TNI"],      #replaces link in header with just date and tni
    parse_dates=["Date"],       #tells pandas these are dates 
    na_values=[-99.99,-9999],
)

#read nc file
#xr.open_dataset(tni_historic_nc)


In [4]:
#uncomment this to see the data dims and range

display(tni_historic_df)
display(oni_historic_df)
display(roni_historic_df)

,Date,TNI
0,1870-01-01,NaN
1,1870-02-01,NaN
2,1870-03-01,0.834
3,1870-04-01,1.127
4,1870-05-01,0.863
...,...,...
1879,2026-08-01,NaN
1880,2026-09-01,NaN
1881,2026-10-01,NaN
1882,2026-11-01,NaN


,YEAR,MON/MMM,NINO34_MEAN,NINO34_CLIM,NINO34_ANOM,ONI,PHASE
0,1850,1,25.99,NaN,NaN,NaN,M
1,1850,2,25.36,NaN,NaN,NaN,M
2,1850,3,25.09,NaN,NaN,NaN,M
3,1850,4,26.23,NaN,NaN,NaN,M
4,1850,5,27.41,NaN,NaN,NaN,M
...,...,...,...,...,...,...,...
2119,2026,8,29.04,26.87,2.17,NaN,M
2120,2026,9,NaN,26.71,NaN,NaN,M
2121,2026,10,NaN,26.71,NaN,NaN,M
2122,2026,11,NaN,26.71,NaN,NaN,M


,YEAR,MON,NINO34_MEAN,NINO34_CLIM,TROP_MEAN,TROP_CLIM,RONI,PHASE
0,1850,1,25.99,26.61,26.65,27.09,NaN,M
1,1850,2,25.36,26.80,26.84,27.25,-1.14,L
2,1850,3,25.09,27.31,26.84,27.59,-1.51,L
3,1850,4,26.23,27.90,27.07,27.85,-1.16,L
4,1850,5,27.41,27.98,27.36,27.78,-0.69,L
...,...,...,...,...,...,...,...,...
2115,2026,4,28.23,27.90,28.27,27.85,-0.04,N
2116,2026,5,28.81,27.98,28.23,27.78,0.51,E
2117,2026,6,29.17,27.76,27.94,27.37,1.00,E
2118,2026,7,29.06,27.31,27.54,26.92,1.36,E


In [5]:
#create single date column for ONI and for RONI
#align date's of the data 

'''
ONI and RONI index have separate year and month columns
this TNI dataset also only goes back to 1870
'''

#form a single date column for oni 
oni_historic_df['Date']=pd.to_datetime(
    {
        'year':oni_historic_df['YEAR'],
        'month':oni_historic_df['MON/MMM'],
        'day':1
    }
)


#form a single date colum for roni 
roni_historic_df['Date']=pd.to_datetime(
    {
        'year':roni_historic_df['YEAR'],
        'month':roni_historic_df['MON'],
        'day':1
    }
)

In [6]:
#validate monthly index dates and values before merging
#validate monthly index dates and values before merging
def validate_monthly_index(df: pd.DataFrame, index_name: str):
    required = ["Date", index_name]
    missing_columns = [column for column in required if column not in df.columns]
    if missing_columns:
        raise ValueError(f"{index_name}: missing columns {missing_columns}")
    if df.empty:
        raise ValueError(f"{index_name}: the source table has no rows.")

    df = df.copy()
    dates = pd.to_datetime(df["Date"], errors="coerce")
    bad_dates = dates.isna() | dates.dt.day.ne(1)
    if bad_dates.any():
        examples = df.loc[bad_dates, "Date"].head(5).tolist()
        raise ValueError(
            f"{index_name}: invalid or non-first-of-month dates: {examples}"
        )
    df["Date"] = dates

    values = pd.to_numeric(df[index_name], errors="coerce")
    bad_values = (
        (df[index_name].notna() & values.isna())
        | values.isin([float("inf"), float("-inf")])
    )
    if bad_values.any():
        examples = df.loc[bad_values, index_name].head(5).tolist()
        raise ValueError(f"{index_name}: invalid index values: {examples}")
    df[index_name] = values

    identical_rows = df.duplicated()
    if identical_rows.any():
        print(f"{index_name}: removing {int(identical_rows.sum())} identical rows.")
        df = df.loc[~identical_rows].copy()

    conflicting_dates = df["Date"].duplicated(keep=False)
    if conflicting_dates.any():
        examples = df.loc[conflicting_dates, "Date"].astype(str).unique()[:5]
        raise ValueError(
            f"{index_name}: conflicting records share a month: {examples.tolist()}"
        )

    return df.sort_values("Date").reset_index(drop=True)


oni_historic_df = validate_monthly_index(oni_historic_df, "ONI")
roni_historic_df = validate_monthly_index(roni_historic_df, "RONI")
tni_historic_df = validate_monthly_index(tni_historic_df, "TNI")

In [7]:
# uncomment this to see how the dataframes were modified
display(tni_historic_df)
display(oni_historic_df)
display(roni_historic_df)

,Date,TNI
0,1870-01-01,NaN
1,1870-02-01,NaN
2,1870-03-01,0.834
3,1870-04-01,1.127
4,1870-05-01,0.863
...,...,...
1879,2026-08-01,NaN
1880,2026-09-01,NaN
1881,2026-10-01,NaN
1882,2026-11-01,NaN


,YEAR,MON/MMM,NINO34_MEAN,NINO34_CLIM,NINO34_ANOM,ONI,PHASE,Date
0,1850,1,25.99,NaN,NaN,NaN,M,1850-01-01
1,1850,2,25.36,NaN,NaN,NaN,M,1850-02-01
2,1850,3,25.09,NaN,NaN,NaN,M,1850-03-01
3,1850,4,26.23,NaN,NaN,NaN,M,1850-04-01
4,1850,5,27.41,NaN,NaN,NaN,M,1850-05-01
...,...,...,...,...,...,...,...,...
2119,2026,8,29.04,26.87,2.17,NaN,M,2026-08-01
2120,2026,9,NaN,26.71,NaN,NaN,M,2026-09-01
2121,2026,10,NaN,26.71,NaN,NaN,M,2026-10-01
2122,2026,11,NaN,26.71,NaN,NaN,M,2026-11-01


,YEAR,MON,NINO34_MEAN,NINO34_CLIM,TROP_MEAN,TROP_CLIM,RONI,PHASE,Date
0,1850,1,25.99,26.61,26.65,27.09,NaN,M,1850-01-01
1,1850,2,25.36,26.80,26.84,27.25,-1.14,L,1850-02-01
2,1850,3,25.09,27.31,26.84,27.59,-1.51,L,1850-03-01
3,1850,4,26.23,27.90,27.07,27.85,-1.16,L,1850-04-01
4,1850,5,27.41,27.98,27.36,27.78,-0.69,L,1850-05-01
...,...,...,...,...,...,...,...,...,...
2115,2026,4,28.23,27.90,28.27,27.85,-0.04,N,2026-04-01
2116,2026,5,28.81,27.98,28.23,27.78,0.51,E,2026-05-01
2117,2026,6,29.17,27.76,27.94,27.37,1.00,E,2026-06-01
2118,2026,7,29.06,27.31,27.54,26.92,1.36,E,2026-07-01


In [8]:
#combine dataframes of oni and roni
#outer joins on the new column (key) DATE

oni_roni_historic_df=pd.merge(
    left=oni_historic_df[["Date","ONI"]], 
    right=roni_historic_df[["Date","RONI"]],
    on='Date', 
    how='outer',
    validate='one_to_one',
    sort=True
)

#combine oni/roni df with tni

oni_roni_tni_historic_df=pd.merge(
    left=oni_roni_historic_df,
    right=tni_historic_df,
    on='Date',
    how='outer',
    validate='one_to_one',
    sort=True,
)



In [9]:
''' 
Here is the final product
a dataframe of all three indicies
aligned by date
'''
oni_roni_tni_historic_df["Date"] = pd.to_datetime(
    oni_roni_tni_historic_df["Date"]
)
oni_roni_tni_historic_df["Year"] = oni_roni_tni_historic_df["Date"].dt.year


oni_roni_tni_historic_df.to_csv(output_roni_oni_tni_indicies, index=False)

#uncomment to see
display(oni_roni_tni_historic_df)


,Date,ONI,RONI,TNI,Year
0,1850-01-01,NaN,NaN,NaN,1850
1,1850-02-01,NaN,-1.14,NaN,1850
2,1850-03-01,NaN,-1.51,NaN,1850
3,1850-04-01,NaN,-1.16,NaN,1850
4,1850-05-01,NaN,-0.69,NaN,1850
...,...,...,...,...,...
2119,2026-08-01,NaN,NaN,NaN,2026
2120,2026-09-01,NaN,NaN,NaN,2026
2121,2026-10-01,NaN,NaN,NaN,2026
2122,2026-11-01,NaN,NaN,NaN,2026


In [ ]:
#record index source snapshots and usable coverage
source_specs = [
    ("ONI", oni_historic, oni_historic_df, "ERSSTv6",
     "https://bmcnoldy.earth.miami.edu/tropics/oni/ONI_NINO34_v6.txt"),
    ("RONI", roni_historic, roni_historic_df, "ERSSTv6",
     "https://bmcnoldy.earth.miami.edu/tropics/roni/RONI_NINO34_v6.txt"),
    ("TNI", tni_historic, tni_historic_df, "HadiSST",
     "https://psl.noaa.gov/data/timeseries/month/data/tni.long.csv"),
]

recorded_utc = pd.Timestamp.now(tz="UTC").isoformat()
source_rows = []
for index_name, path, frame, version, url in source_specs:
    file_info = path.stat()
    valid_dates = frame.loc[frame[index_name].notna(), "Date"]
    expected_months = len(pd.period_range(
        start=str(frame["Date"].min()),
        end=str(frame["Date"].max()),
        freq="M",
    ))

    source_rows.append({
        "index": index_name,
        "source_url": url,
        "source_file": path.name,
        "version_note": version,
        "bytes": file_info.st_size,
        "file_modified_utc": pd.Timestamp(
            file_info.st_mtime, unit="s", tz="UTC"
        ).isoformat(),
        "recorded_utc": recorded_utc,
        "sha256": sha256(path.read_bytes()).hexdigest(),
        "first_month": frame["Date"].min(),
        "last_month": frame["Date"].max(),
        "expected_months": expected_months,
        "source_rows": len(frame),
        "absent_month_rows": expected_months - len(frame),
        "valid_months": len(valid_dates),
        "missing_values": int(frame[index_name].isna().sum()),
        "first_valid": valid_dates.min(),
        "last_valid": valid_dates.max(),
    })

index_source_inventory = pd.DataFrame(source_rows)
report_dir = Path("../outputs")
report_dir.mkdir(parents=True, exist_ok=True)
inventory_path = report_dir / "enso_index_source_inventory.csv"
index_source_inventory.to_csv(
    inventory_path, index=False, date_format="%Y-%m-%d"
)
display(index_source_inventory)

,index,source_url,source_file,version_note,bytes,file_modified_utc,recorded_utc,sha256,first_month,last_month,expected_months,source_rows,absent_month_rows,valid_months,missing_values,first_valid,last_valid
0,ONI,https://bmcnoldy.earth.miami.edu/tropics/oni/O...,ONI_NINO34_v6.txt,ERSSTv6,157987,2026-09-30T20:19:09.470464706+00:00,2026-11-02T17:43:02.584003+00:00,a2a5e13aefd7f36c753c67c6b4226bfe577135dcc5c4b3...,1850-01-01,2026-12-01,2124,2124,0,1926,198,1866-02-01,2026-07-01
1,RONI,https://bmcnoldy.earth.miami.edu/tropics/roni/...,RONI_NINO34_v6.txt,ERSSTv6,178977,2026-09-30T20:18:52.688497782+00:00,2026-11-02T17:43:02.584003+00:00,0fb181e2fcae2c0753987ffa8501cfc104d08944921717...,1850-01-01,2026-08-01,2120,2120,0,2118,2,1850-02-01,2026-07-01
2,TNI,https://psl.noaa.gov/data/timeseries/month/dat...,tni.long.csv,not specified in CSV,39642,2026-09-30T19:37:47.462033987+00:00,2026-11-02T17:43:02.584003+00:00,024b87eaa64719bff8de4c40c8a3a3d9c5bd7b0c0c75e4...,1870-01-01,2026-12-01,1884,1884,0,1875,9,1870-03-01,2026-05-01
